# Phase 1: Supervised Learning – Random Forest

## 1️⃣ Model Selection Rationale
- **Why Random Forest**  
  - Handles nonlinear relationships in medical data  
  - Reduces overfitting compared to a single tree  
  - Works well with small tabular datasets (~583 rows)  
  - Provides feature importance for interpretability  

- **Strengths**
  - Robust to noise
  - Handles nonlinear patterns
  - Minimal scaling required
  - Feature importance visualization

- **Weaknesses**
  - Less interpretable than Logistic Regression
  - Higher computational cost
  - Can overfit if not tuned

---

## 2️⃣ Data Preparation
- Load the dataset (`Dataset/preprocessed_data.csv`)
- Encode categorical features (Gender → 0/1)
- Separate `X` (features) and `y` (target)
- Train/Test split (e.g., 80/20)
- Optional: check class balance

1️⃣ Model Selection Rationale

Why Random Forest?

Dataset: 583 samples, mostly numeric features.

Problem: Binary classification (Dataset column: liver disease yes/no).

Relationships between medical variables may be nonlinear.

Random Forest handles nonlinear patterns and reduces overfitting compared to a single decision tree.

Provides feature importance, which is great for interpretability in medical data.

Strengths:

Robust to noise

Handles nonlinear relationships

Works well without scaling

Can generate feature importance plots

Weaknesses:

Less interpretable than Logistic Regression

Higher computational cost than a single tree

Can still overfit if not tuned

Use in Project:
We are training the model to predict liver disease based on patient features. Later, this prediction will inform the advice system (“Consult a hepatologist” or “Routine check-up”).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("Dataset/preprocessed_data.csv")

# Check first rows
df.head()

# Features and target
X = df.drop("Dataset", axis=1)
y = df["Dataset"]

# Encode Gender if not already numeric (0=Female, 1=Male)
# (Looks like your CSV already has 0/1, so skip encoding)

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Optional: scale features (Random Forest doesn't strictly need it)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Baseline model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# Evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# GridSearchCV with 5-fold cross-validation
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',  # Emphasize recall/precision for medical data
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best CV F1 Score:", grid_rf.best_score_)

In [ ]:
# Best model
best_rf = grid_rf.best_estimator_

# Feature importance
importances = best_rf.feature_importances_
features = X.columns
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)

# Plot
sns.barplot(x=feat_imp, y=feat_imp.index)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='f1')
print(f"5-Fold CV F1 Score: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")